# Stage 2A operational retrieval smoke
Integration and latency smoke only; this notebook makes no ranking-quality claim.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")


def resolve_root(root, marker, max_depth=3):
    root = Path(root)
    matches = []
    frontier = [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if (current / marker).is_file():
            matches.append(current)
            continue
        if depth < max_depth and current.is_dir():
            frontier.extend(
                (child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir()
            )
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one root containing {marker} below {root}; found {matches}"
        )
    return matches[0]


STAGE1_ROOT = resolve_root(
    "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle", "stage1_summary.json"
)
STAGE1B_ROOT = resolve_root(
    "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    "stage1b_summary.json",
)
STAGE1E_ROOT = resolve_root(
    "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    "language_path_contract.json",
)
CLIP_ROOT = resolve_root(
    "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32", "manifests/asset_manifest.json"
)
OPUS_ROOT = resolve_root(
    "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en", "manifests/asset_manifest.json"
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_stage2a_operational_runtime")
print(
    {
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
    }
)

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
from triage_eg.retrieval.stage2 import OperationalRetrievalRuntime, config_from_yaml

CONFIG = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=OUTPUT_ROOT,
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=COMMIT,
)
runtime = OperationalRetrievalRuntime(CONFIG).load()
print(json.dumps(runtime.preflight, indent=2))

In [ ]:
print(json.dumps(runtime.runtime_manifest(), indent=2, default=str))

In [ ]:
from triage_eg.retrieval.stage2 import QueryRequest

en_results = runtime.search_many(
    [
        QueryRequest("smoke_en_cooking", "a person cooking in a kitchen", "en", 20),
        QueryRequest("smoke_en_red_car", "a red car", "en", 20),
    ]
)
en_result = en_results[0]
print([(item.query_id, item.language_resolution, len(item.ranked_frames)) for item in en_results])

In [ ]:
vi_results = runtime.search_many(
    [
        QueryRequest("smoke_vi_cooking", "một người đang nấu ăn trong bếp", "vi", 20),
        QueryRequest("smoke_vi_red_car", "một chiếc ô tô màu đỏ", "vi", 20),
    ]
)
vi_result = vi_results[0]
print(
    [
        (item.query_id, item.language_resolution, item.encoding["translated_text"])
        for item in vi_results
    ]
)

In [ ]:
print({"EN": [item.encoding for item in en_results], "VI": [item.encoding for item in vi_results]})

In [ ]:
print(
    {
        "EN_ms": [item.latencies_ms for item in en_results],
        "VI_ms": [item.latencies_ms for item in vi_results],
    }
)

In [ ]:
print((en_result.output_root / "kis_candidates.csv").read_text().splitlines()[:6])

In [ ]:
manifest = runtime.runtime_manifest()
print("OPERATIONAL_RETRIEVAL_RUNTIME_STATUS = READY_FOR_REAL_SMOKE")
print("RANKING_QUALITY_STATUS = UNCHANGED_FROM_FROZEN_BASELINE")
runtime.close()

In [ ]:
from triage_eg.retrieval.stage2 import create_stage2_report_bundle

ZIP_PATH = Path("/kaggle/working/triage_eg_stage2a_operational_runtime_reports.zip")
create_stage2_report_bundle(OUTPUT_ROOT, ZIP_PATH)
print("DOWNLOAD ZIP:", ZIP_PATH, "size_bytes=", ZIP_PATH.stat().st_size)